<a href="https://colab.research.google.com/github/khovan123/cropstate/blob/main/notebooks/cropstate_colab_github_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CROPSTATE Colab Workflow

Notebook này dùng cho workflow:

- Code clone từ GitHub
- Dataset đọc từ Google Drive: `MyDrive/CROPSTATE_DATASET`
- Knowledge base đọc từ Google Drive: `MyDrive/CROPSTATE_KNOWLEDGE_BASE`
- Training outputs lưu vào Google Drive: `MyDrive/CROPSTATE_RESULTS`

Trước khi chạy training, vào `Runtime > Change runtime type` và chọn GPU.

In [7]:
# Sửa REPO_URL thành GitHub repo thật của bạn.
REPO_URL = "https://github.com/khovan123/cropstate"
PROJECT_DIR = "/content/CROPSTATE_Full_Research_Package"

DATA_ROOT = "/content/drive/MyDrive/CROPSTATE_DATASET"
KNOWLEDGE_ROOT = "/content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE"
RESULTS_ROOT = "/content/drive/MyDrive/CROPSTATE_RESULTS"

print("REPO_URL:", REPO_URL)
print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_ROOT:", DATA_ROOT)
print("KNOWLEDGE_ROOT:", KNOWLEDGE_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)

REPO_URL: https://github.com/khovan123/cropstate
PROJECT_DIR: /content/CROPSTATE_Full_Research_Package
DATA_ROOT: /content/drive/MyDrive/CROPSTATE_DATASET
KNOWLEDGE_ROOT: /content/drive/MyDrive/CROPSTATE_KNOWLEDGE_BASE
RESULTS_ROOT: /content/drive/MyDrive/CROPSTATE_RESULTS


## 1. Mount Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Clone hoặc update repo từ GitHub

In [4]:
import os
import subprocess

if not os.path.exists(PROJECT_DIR):
    subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
else:
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)

os.chdir(PROJECT_DIR)
print("Current directory:", os.getcwd())

Current directory: /content/CROPSTATE_Full_Research_Package


## 3. Cài dependencies

In [5]:
!pip install -r requirements.txt
!pip install -e .

Obtaining file:///content/CROPSTATE_Full_Research_Package
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for cropstate-research (pyproject.toml) ... done
  Created wheel for cropstate-research: filename=cropstate_research-0.1.0-0.editable-py3-none-any.whl size=1383 sha256=26a109959c066a78dc139ca7b88ba7058128ee79a12c71f9b814a690a640d6d8
  Stored in directory: /tmp/pip-ephem-wheel-cache-5vsw8_o2/wheels/37/37/8f/89550734e79d9e7fc68e041f688e3bc8f62ff3b93e9d2c762c
Successfully built cropstate-research


## 4. Kiểm tra dataset và knowledge base trên Drive

In [8]:
!ls -lah "{DATA_ROOT}"
!ls -lah "{KNOWLEDGE_ROOT}"

total 24K
drwx------ 2 root root 4.0K Jul  5 09:28 01_establishment
drwx------ 2 root root 4.0K Jul  6 16:19 02_tillering
drwx------ 2 root root 4.0K Jul  6 16:19 03_stem_booting
drwx------ 2 root root 4.0K Jul  6 16:19 04_reproductive
drwx------ 2 root root 4.0K Jul  5 11:04 05_grain_filling
drwx------ 2 root root 4.0K Jul  5 11:04 06_ripening
total 30K
-rw------- 1 root root 8.0K Jul  5 21:23 CROPSTATE_KB_Evaluation_Report.docx
-rw------- 1 root root  22K Jul  5 21:22 CROPSTATE_Sample_Knowledge_Base.xlsx


Dataset nên có cấu trúc:

```text
CROPSTATE_DATASET/
  01_establishment/
  02_tillering/
  03_stem_booting/
  04_reproductive/
  05_grain_filling/
  06_ripening/
```

Knowledge base nên có `CROPSTATE_Sample_Knowledge_Base.xlsx` hoặc CSV exports trong `CROPSTATE_KNOWLEDGE_BASE/`.

## 5. Build manifest pilot từ folder dataset

In [9]:
!PYTHONPATH=src python scripts/build_stage_manifest.py \
  --data-root "{DATA_ROOT}" \
  --output data/stage_folder_manifest.csv

Wrote 557 rows to data/stage_folder_manifest.csv
macro_stage
tillering        150
ripening         148
grain_filling    125
stem_booting      54
reproductive      46
establishment     34


## 6. Convert image manifest từ knowledge base

Nếu `Image_Manifest_Template` vẫn là template placeholder, `data/image_manifest.csv` sẽ rỗng và row bị loại sẽ nằm trong `data/image_manifest_excluded.csv`.

In [10]:
!PYTHONPATH=src python scripts/convert_image_manifest.py \
  --knowledge-root "{KNOWLEDGE_ROOT}" \
  --data-root "{DATA_ROOT}" \
  --output data/image_manifest.csv

Wrote training manifest: data/image_manifest.csv (0 rows)
Excluded rows: 1
Exact duplicate image rows by sha256: 0


## 7. Convert knowledge chunks

In [11]:
!PYTHONPATH=src python scripts/convert_knowledge_base.py \
  --knowledge-root "{KNOWLEDGE_ROOT}" \
  --output data/knowledge_chunks.jsonl

Wrote 14 chunks to data/knowledge_chunks.jsonl


## 8. Audit manifest pilot từ folder

In [ ]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum

Rows: 557

Class counts:
 macro_stage
tillering        150
ripening         148
grain_filling    125
stem_booting      54
reproductive      46
establishment     34
Name: count, dtype: int64

Split counts:
 split
unassigned    557
Name: count, dtype: int64

parent_image_id leakage groups: 0

field_id leakage groups: 0

capture_session leakage groups: 0

Unlabeled rows: 0

Missing image files: 0

Non-training labels in manifest: 0


## 9. Audit manifest từ sheet nếu đã có dữ liệu thật

Chỉ dùng manifest này để train khi `data/image_manifest.csv` có row thật và audit pass.

In [ ]:
!PYTHONPATH=src python scripts/audit_dataset.py \
  --manifest data/image_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --checksum

## 10. Fine-tune bằng manifest pilot từ folder

Output được lưu thẳng vào Google Drive.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --output "{RESULTS_ROOT}/vision_resnet18_finetune"

## 11. Fine-tune bằng manifest từ sheet

Chỉ chạy cell này nếu `data/image_manifest.csv` đã có row thật.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest data/image_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --output "{RESULTS_ROOT}/vision_resnet18_manifest_finetune"

## 12. Fine-tune tiếp từ checkpoint cũ

Dùng khi đã có `best_checkpoint.pt` từ lần train trước.

In [ ]:
!PYTHONPATH=src python scripts/train_vision.py \
  --manifest data/stage_folder_manifest.csv \
  --data-root "{DATA_ROOT}" \
  --config configs/vision.yaml \
  --resume-checkpoint "{RESULTS_ROOT}/vision_resnet18_finetune/best_checkpoint.pt" \
  --freeze-backbone-epochs 0 \
  --learning-rate 0.0001 \
  --backbone-learning-rate 0.00001 \
  --output "{RESULTS_ROOT}/vision_resnet18_finetune_round2"

## 13. Xem output đã lưu trên Drive

In [ ]:
!ls -lah "{RESULTS_ROOT}/vision_resnet18_finetune"

## 14. Test một ảnh upload từ máy

In [ ]:
from google.colab import files

uploaded = files.upload()
image_path = next(iter(uploaded))
print("Uploaded:", image_path)

In [ ]:
!PYTHONPATH=src python scripts/predict_image.py \
  --checkpoint "{RESULTS_ROOT}/vision_resnet18_finetune/best_checkpoint.pt" \
  --image "{image_path}"

## Ghi chú

- GitHub chỉ lưu code và metadata nhỏ.
- Google Drive lưu dataset, knowledge base, checkpoint và kết quả training.
- Không commit các file `.pt` lên GitHub.